This notebooks includes all experiments included in Section PRNU based source camera identification of the article. Namely it allows the user to perform source camera identification using the python implementation extracted from https://github.com/sim-pez/prnu/tree/main of the he Binghamton Toolbox originally implemented in Matlab. 

This code provides the following results:
1. PRNU and natural images JPG and raw, roc curves, FPR/FNR and search for better threshold.
2. PRNU and natural images JPG, spce and pearson correlation violin diagrams.
3. PRNU and natural images raw, spce and pearson correlation violin diagrams.


## PRNU and natural images JPG and raw roc curve

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu
import matplotlib.pyplot as plt
import argparse

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))  # ['A1', 'A2', ...]
jpeg_ext = 'jpg'

flat_images, flat_devices = [], []
natural_images, natural_devices = [], []

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'flat', 'jpg')
    natural_dir = os.path.join(base_dir, tag, 'natural', 'jpg')
    
    # Flats
    for img_path in glob(os.path.join(flat_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        flat_images.append(img_path)
        flat_devices.append(device)
        
    # Naturals
    for img_path in glob(os.path.join(natural_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        natural_images.append(img_path)
        natural_devices.append(device)

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)
natural_images = np.array(natural_images)
natural_devices = np.array(natural_devices)

In [ ]:
print('Computing fingerprints')
fingerprint_devices = sorted(np.unique(flat_devices))

k = []
for device in fingerprint_devices:
    print(device)
    imgs = []
    for img_path in flat_images[flat_devices == device]:
        im = np.asarray(Image.open(img_path))
        if im.dtype != np.uint8 or im.ndim != 3:
            print('Skipping image:', img_path)
            continue
        imgs.append(prnu.cut_ctr(im, (512, 512, 3)))
    k.append(prnu.extract_multiple_aligned(imgs, processes=1))

k = np.stack(k, 0)

In [ ]:
print('Computing residuals')
w = []
for img_path in natural_images:
    im = np.asarray(Image.open(img_path))
    w.append(prnu.cut_ctr(im, (512, 512, 3)))

w = np.stack([prnu.extract_single(img) for img in w], 0)

In [ ]:
matching_pce = []
mismatching_pce = []

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]
    for n_idx, natural_w in enumerate(w):
        device_nat = natural_devices[n_idx]
        cc2d = prnu.crosscorr_2d(fingerprint_k, natural_w)
        pce_value = prnu.pce(cc2d)['pce']
        
        if device_fp == device_nat:
            matching_pce.append(pce_value)
        else:
            mismatching_pce.append(pce_value)

In [ ]:
np.savez('roc_data_jpeg.npz', 
         matching_pce=matching_pce, 
         mismatching_pce=mismatching_pce,
         threshold=60)

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu_raw
import matplotlib.pyplot as plt
import argparse

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

In [ ]:
def split_bayer(im):
    R  = im[0::2, 0::2]
    G1 = im[0::2, 1::2]
    G2 = im[1::2, 0::2]
    B  = im[1::2, 1::2]
    return [R, G1, G2, B]

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))

flat_images, flat_devices = [], []
natural_images, natural_devices = [], []

raw_exts = ['cr3', 'cr2', 'arw']

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'flat', 'raw')
    natural_dir = os.path.join(base_dir, tag, 'natural', 'raw')
    
    # Flats
    for ext in raw_exts:
        for img_path in glob(os.path.join(flat_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            flat_images.append(img_path)
            flat_devices.append(device)
        
    # Naturals
    for ext in raw_exts:
        for img_path in glob(os.path.join(natural_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            natural_images.append(img_path)
            natural_devices.append(device)

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)
natural_images = np.array(natural_images)
natural_devices = np.array(natural_devices)

In [ ]:
def open_image(img_path):
    img = rawpy.imread(img_path)
    return img.raw_image_visible

In [ ]:
print('Computing fingerprints')
fingerprint_devices = sorted(np.unique(flat_devices))

k = []  # [num_devices, 4, H, W]

for device in fingerprint_devices:
    print(device)
    channel_imgs = [[], [], [], []]  # R, G1, G2, B
    
    for img_path in flat_images[flat_devices == device]:
        im = np.asarray(open_image(img_path))
        channels = split_bayer(im)
        
        for c in range(4):
            channel_imgs[c].append(prnu_raw.cut_ctr(channels[c], (512, 512)))
    
    k_device = []
    for c in range(4):
        k_c = prnu_raw.extract_multiple_aligned(channel_imgs[c], processes=1)
        k_device.append(k_c)
    
    k.append(k_device)

k = np.array(k)

In [ ]:
print('Computing residuals')
w = []  # shape: [num_images, 4, H, W]

for img_path in natural_images:
    im = np.asarray(open_image(img_path))
    channels = split_bayer(im)
    
    w_channels = []
    for c in range(4):
        cut = prnu_raw.cut_ctr(channels[c], (512, 512))
        w_c = prnu_raw.extract_single(cut)
        w_channels.append(w_c)
    
    w.append(w_channels)

w = np.array(w)

In [ ]:
matching_pce_ch = [[] for _ in range(4)]
mismatching_pce_ch = [[] for _ in range(4)]

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]
    
    for n_idx, natural_w in enumerate(w):
        device_nat = natural_devices[n_idx]
        
        for c in range(4):
            cc2d = prnu_raw.crosscorr_2d(fingerprint_k[c], natural_w[c])
            pce_value = prnu_raw.pce(cc2d)['pce']
            
            if device_fp == device_nat:
                matching_pce_ch[c].append(pce_value)
            else:
                mismatching_pce_ch[c].append(pce_value)

In [ ]:
channel_names = ['R', 'G1', 'G2', 'B']

save_dict = {}

for c in range(4):
    save_dict[f'matching_pce_{channel_names[c]}'] = np.array(matching_pce_ch[c])
    save_dict[f'mismatching_pce_{channel_names[c]}'] = np.array(mismatching_pce_ch[c])

save_dict['threshold'] = 60

np.savez('roc_data_per_channel.npz', **save_dict)

In [ ]:
for c in range(4):
    y_true = [1]*len(matching_pce_ch[c]) + [0]*len(mismatching_pce_ch[c])
    y_scores = matching_pce_ch[c] + mismatching_pce_ch[c]
    

    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    target_fpr = 1e-4

    idx = np.argmin(np.abs(fpr - target_fpr))
    threshold_fpr = thresholds[idx]
    
    print("Channel:", c)

    print("Threshold for FPR≈1e-4%:", threshold_fpr)

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve, auc

data_channels = np.load('roc_data_per_channel.npz')
data_jpeg = np.load('roc_data_jpeg.npz')

channel_names = ['R', 'G1', 'G2', 'B']

colors = sns.color_palette("rocket", n_colors=5)

plt.figure(figsize=(8, 6))


for i, ch in enumerate(channel_names):
    matching = data_channels[f'matching_pce_{ch}']
    mismatching = data_channels[f'mismatching_pce_{ch}']
    
    y_true = np.concatenate([np.ones_like(matching), np.zeros_like(mismatching)])
    scores = np.concatenate([matching, mismatching])
    
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], label=f'{ch} (AUC = {roc_auc:.3f})')

matching = data_jpeg['matching_pce']
mismatching = data_jpeg['mismatching_pce']

y_true = np.concatenate([np.ones_like(matching), np.zeros_like(mismatching)])
scores = np.concatenate([matching, mismatching])
fpr_j, tpr_j, _ = roc_curve(y_true, scores)
auc_j = auc(fpr_j, tpr_j)
plt.plot(fpr_j, tpr_j, '--', color=colors[4], linewidth=2, label=f'JPG (AUC = {auc_j:.3f})')

# Random classifier
plt.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Random')


plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.xlabel('False Positive Rate', fontsize = 16)
plt.ylabel('True Positive Rate', fontsize = 16)
plt.title('Receiver Operating Characteristic (ROC) Curve \n PRNU based source camera identification', fontsize = 16)
plt.legend(fontsize = 11)
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for i, ch in enumerate(channel_names):
    matching = data_channels[f'matching_pce_{ch}']
    mismatching = data_channels[f'mismatching_pce_{ch}']
    
    y_true = np.concatenate([np.ones_like(matching), np.zeros_like(mismatching)])
    scores = np.concatenate([matching, mismatching])
    
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], label=f'{ch} (AUC = {roc_auc:.3f})')

matching = data_jpeg['matching_pce']
mismatching = data_jpeg['mismatching_pce']

y_true = np.concatenate([np.ones_like(matching), np.zeros_like(mismatching)])
scores = np.concatenate([matching, mismatching])
fpr_j, tpr_j, _ = roc_curve(y_true, scores)
auc_j = auc(fpr_j, tpr_j)
plt.plot(fpr_j, tpr_j, '--', color=colors[4], linewidth=2, label=f'JPG (AUC = {auc_j:.3f})')


plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.xlabel('False Positive Rate', fontsize = 16)
plt.ylabel('True Positive Rate', fontsize = 16)
plt.title('Receiver Operating Characteristic (ROC) Curve \n PRNU based source camera identification', fontsize = 16)
plt.legend(fontsize = 11)
plt.xlim(0,0.1)
plt.ylim(0.9,1)
plt.grid(True)
plt.show()

In [ ]:
data = np.load('roc_data_per_channel.npz')

channel_names = ['R', 'G1', 'G2', 'B']
threshold = data['threshold'].item()

results = {}

for ch in channel_names:
    matching = data[f'matching_pce_{ch}']
    mismatching = data[f'mismatching_pce_{ch}']
    
    FN = np.sum(matching < threshold)
    FP = np.sum(mismatching >= threshold)
    TP = np.sum(matching >= threshold)
    TN = np.sum(mismatching < threshold)
    
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    FNR = FN / (FN + TP) if (FN + TP) > 0 else 0
    
    results[ch] = {
        'FP': FP,
        'FN': FN,
        'TP': TP,
        'TN': TN,
        'FPR': FPR,
        'FNR': FNR
    }

for ch, res in results.items():
    print(f"Canal {ch}:")
    print(f"  FP: {res['FP']}, FN: {res['FN']}, TP: {res['TP']}, TN: {res['TN']}")
    print(f"  FPR: {res['FPR']:.4f}, FNR: {res['FNR']:.4f}")

In [ ]:
data = np.load('roc_data_jpeg.npz')

matching = data['matching_pce']
mismatching = data['mismatching_pce']
threshold = data['threshold'].item()

FN = np.sum(matching < threshold)
FP = np.sum(mismatching >= threshold)

TP = np.sum(matching >= threshold)
TN = np.sum(mismatching < threshold)

FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
FNR = FN / (FN + TP) if (FN + TP) > 0 else 0

print(f"Threshold: {threshold}")
print(f"TP: {TP}")
print(f"TN: {TN}")
print(f"FP: {FP}")
print(f"FN: {FN}")
print(f"FPR: {FPR:.4f}")
print(f"FNR: {FNR:.4f}")

## PRNU and natural images JPG violin diagrams

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu
import matplotlib.pyplot as plt
import argparse
from scipy.stats import pearsonr

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

from collections import defaultdict

import numpy as np

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))  # ['A1', 'A2', ...]
jpeg_ext = 'jpg'

flat_images, flat_devices = [], []
natural_images, natural_devices = [], []

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'flat', 'jpg')
    natural_dir = os.path.join(base_dir, tag, 'natural', 'jpg')
    
    # Flats
    for img_path in glob(os.path.join(flat_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        flat_images.append(img_path)
        flat_devices.append(device)
        
    # Naturals
    for img_path in glob(os.path.join(natural_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        natural_images.append(img_path)
        natural_devices.append(device)

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)
natural_images = np.array(natural_images)
natural_devices = np.array(natural_devices)

In [ ]:
print('Computing fingerprints')
fingerprint_devices = sorted(np.unique(flat_devices))

k = []
for device in fingerprint_devices:
    print(device)
    imgs = []
    for img_path in flat_images[flat_devices == device]:
        im = np.asarray(Image.open(img_path))
        if im.dtype != np.uint8 or im.ndim != 3:
            print('Skipping image:', img_path)
            continue
        imgs.append(prnu.cut_ctr(im, (512, 512, 3)))
    k.append(prnu.extract_multiple_aligned(imgs, processes=1))

k = np.stack(k, 0)

In [ ]:
print('Computing residuals')
w = []
for img_path in natural_images:
    im = np.asarray(Image.open(img_path))
    w.append(prnu.cut_ctr(im, (512, 512, 3)))

w = np.stack([prnu.extract_single(img) for img in w], 0)

In [ ]:
matching_cases = []
mismatching_cases = []

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]

    for n_idx, natural_w in enumerate(w):
        device_nat = natural_devices[n_idx]

        cc2d = prnu.crosscorr_2d(fingerprint_k, natural_w)
        pce_value = prnu.pce(cc2d)['pce']

        case = {
            'pce': pce_value,
            'fingerprint_device': device_fp,
            'natural_device': device_nat,
            'natural_path': natural_images[n_idx]
        }

        if device_fp == device_nat:
            matching_cases.append(case)
        else:
            mismatching_cases.append(case)

In [ ]:
np.savez(
    "spce_prnu_natural_jpg.npz",
    matching_cases=matching_cases,
    mismatching_cases=mismatching_cases
)

In [ ]:
data = np.load("spce_prnu_natural_jpg.npz", allow_pickle=True)

matching_cases = data["matching_cases"].tolist()
mismatching_cases = data["mismatching_cases"].tolist()

data = defaultdict(lambda: {'match': [], 'mismatch': []})

# Matching
for case in matching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['match'].append(case['pce'])

# Mismatching
for case in mismatching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['mismatch'].append(case['pce'])

sensors = sorted(data.keys())

match_values = [data[s]['match'] for s in sensors]
mismatch_values = [data[s]['mismatch'] for s in sensors]


plt.figure(figsize=(14,6))

positions_match = []
positions_mismatch = []

for i in range(len(sensors)):
    positions_match.append(i - 0.15)
    positions_mismatch.append(i + 0.15)

# Matching
vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
for body in vp1['bodies']:
    body.set_facecolor('green')
    body.set_alpha(0.6)

# Mismatching
vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
for body in vp2['bodies']:
    body.set_facecolor('red')
    body.set_alpha(0.6)


plt.xticks(range(len(sensors)), sensors, rotation=45)
plt.yscale('log')
plt.xlabel("Sensor", fontsize =16)
plt.ylabel("sPCE value", fontsize = 16)
plt.title("sPCE distribution per sensor PRNU using JPG images", fontsize = 16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend([plt.Line2D([0],[0], color='green', lw=4),
            plt.Line2D([0],[0], color='red', lw=4)],
           ['Matching', 'Mismatching'], fontsize = 11)

plt.tight_layout()
plt.show()

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))  # ['A1', 'A2', ...]
jpeg_ext = 'jpg'

flat_images, flat_devices = [], []
natural_images, natural_devices = [], []

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'flat', 'jpg')
    natural_dir = os.path.join(base_dir, tag, 'natural', 'jpg')
    
    # Flats
    for img_path in glob(os.path.join(flat_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        flat_images.append(img_path)
        flat_devices.append(device)
        
    # Naturals
    for img_path in glob(os.path.join(natural_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        natural_images.append(img_path)
        natural_devices.append(device)

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)
natural_images = np.array(natural_images)
natural_devices = np.array(natural_devices)

In [ ]:
print('Computing fingerprints')
fingerprint_devices = sorted(np.unique(flat_devices))

k = []
for device in fingerprint_devices:
    print(device)
    imgs = []
    for img_path in flat_images[flat_devices == device]:
        im = np.asarray(Image.open(img_path))
        if im.dtype != np.uint8 or im.ndim != 3:
            print('Skipping image:', img_path)
            continue
        imgs.append(prnu.cut_ctr(im, (512, 512, 3)))
    k.append(prnu.extract_multiple_aligned(imgs, processes=1))

k = np.stack(k, 0)

In [ ]:
print('Computing residuals')
w = []
for img_path in natural_images:
    im = np.asarray(Image.open(img_path))
    w.append(prnu.cut_ctr(im, (512, 512, 3)))

w = np.stack([prnu.extract_single(img) for img in w], 0)

In [ ]:
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return pearsonr(a_flat,b_flat)[0]

In [ ]:
matching_cases = []
mismatching_cases = []

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]

    for n_idx, natural_w in enumerate(w):
        device_nat = natural_devices[n_idx]

        pearson = pearson_corr(fingerprint_k,natural_w)

        case = {
            'pearson': pearson,
            'fingerprint_device': device_fp,
            'natural_device': device_nat,
            'natural_path': natural_images[n_idx]
        }

        if device_fp == device_nat:
            matching_cases.append(case)
        else:
            mismatching_cases.append(case)

In [ ]:
np.savez(
    "pearson_prnu_natural_jpg.npz",
    matching_cases=matching_cases,
    mismatching_cases=mismatching_cases
)

In [ ]:
data = np.load("pearson_prnu_natural_jpg.npz", allow_pickle=True)
matching_cases = data["matching_cases"].tolist()
mismatching_cases = data["mismatching_cases"].tolist()

In [ ]:
data = defaultdict(lambda: {'match': [], 'mismatch': []})

# Matching
for case in matching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['match'].append(case['pearson'])

# Mismatching
for case in mismatching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['mismatch'].append(case['pearson'])

sensors = sorted(data.keys())

match_values = [data[s]['match'] for s in sensors]
mismatch_values = [data[s]['mismatch'] for s in sensors]

plt.figure(figsize=(14,6))

positions_match = []
positions_mismatch = []

for i in range(len(sensors)):
    positions_match.append(i - 0.15)
    positions_mismatch.append(i + 0.15)

# Matching
vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
for body in vp1['bodies']:
    body.set_facecolor('green')
    body.set_alpha(0.6)

# Mismatching
vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
for body in vp2['bodies']:
    body.set_facecolor('red')
    body.set_alpha(0.6)


plt.xticks(range(len(sensors)), sensors, rotation=45)

plt.xlabel("Sensor", fontsize =16)
plt.ylabel("Pearson correlation value", fontsize = 16)
plt.title("Pearson correlation distribution per sensor PRNU in JPG", fontsize = 16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend([plt.Line2D([0],[0], color='green', lw=4),
            plt.Line2D([0],[0], color='red', lw=4)],
           ['Matching', 'Mismatching'], fontsize = 11)

plt.tight_layout()
plt.show()

## PRNU and natural images raw violin diagrams

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu_raw
import matplotlib.pyplot as plt
import argparse

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

from scipy.stats import pearsonr

from collections import defaultdict

In [ ]:
def split_bayer(im):
    R  = im[0::2, 0::2]
    G1 = im[0::2, 1::2]
    G2 = im[1::2, 0::2]
    B  = im[1::2, 1::2]
    return [R, G1, G2, B]

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))

flat_images, flat_devices = [], []
natural_images, natural_devices = [], []

raw_exts = ['cr3', 'cr2', 'arw']

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'flat', 'raw')
    natural_dir = os.path.join(base_dir, tag, 'natural', 'raw')
    
    # Flats
    for ext in raw_exts:
        for img_path in glob(os.path.join(flat_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            flat_images.append(img_path)
            flat_devices.append(device)
        
    # Naturals
    for ext in raw_exts:
        for img_path in glob(os.path.join(natural_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            natural_images.append(img_path)
            natural_devices.append(device)

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)
natural_images = np.array(natural_images)
natural_devices = np.array(natural_devices)

In [ ]:
def open_image(img_path):
    img = rawpy.imread(img_path)
    return img.raw_image_visible

In [ ]:
print('Computing fingerprints')
fingerprint_devices = sorted(np.unique(flat_devices))

k = []  # [num_devices, 4, H, W]

for device in fingerprint_devices:
    print(device)
    channel_imgs = [[], [], [], []]  # R, G1, G2, B
    
    for img_path in flat_images[flat_devices == device]:
        im = np.asarray(open_image(img_path))
        channels = split_bayer(im)
        
        for c in range(4):
            channel_imgs[c].append(prnu_raw.cut_ctr(channels[c], (512, 512)))
    
    k_device = []
    for c in range(4):
        k_c = prnu_raw.extract_multiple_aligned(channel_imgs[c], processes=1)
        k_device.append(k_c)
    
    k.append(k_device)

k = np.array(k) 

In [ ]:
print('Computing residuals')
w = []  # shape: [num_images, 4, H, W]

for img_path in natural_images:
    im = np.asarray(open_image(img_path))
    channels = split_bayer(im)
    
    w_channels = []
    for c in range(4):
        cut = prnu_raw.cut_ctr(channels[c], (512, 512))
        w_c = prnu_raw.extract_single(cut)
        w_channels.append(w_c)
    
    w.append(w_channels)

w = np.array(w)

In [ ]:
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return pearsonr(a_flat,b_flat)[0]

In [ ]:
channel_names = ['R', 'G1', 'G2', 'B']

results = []

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]
    
    for n_idx, natural_w in enumerate(w):
        device_nat = natural_devices[n_idx]
        
        for c in range(4):
            cc2d = prnu_raw.crosscorr_2d(fingerprint_k[c], natural_w[c])
            pce_value = prnu_raw.pce(cc2d)['pce']
            
            pearson = pearson_corr(fingerprint_k[c], natural_w[c])
            
            results.append({
                'pce': pce_value,
                'pearson': pearson,
                'match': int(device_fp == device_nat),
                'fp_device': device_fp,
                'nat_device': device_nat,
                'img_idx': n_idx,
                'channel': c
            })

In [ ]:
np.savez('spce-pearson-prnu-raw-nat.npz', results=results)

In [ ]:
output_dir = "raw-violins-prnu"
os.makedirs(output_dir, exist_ok=True)

data = np.load('spce-pearson-prnu-raw-nat.npz', allow_pickle=True)
results = data['results']

channel_names = ['R', 'G1', 'G2', 'B']

for c, cname in enumerate(channel_names):

    sensor_data_pce = defaultdict(lambda: {'match': [], 'mismatch': []})
    sensor_data_pearson = defaultdict(lambda: {'match': [], 'mismatch': []})

    for case in results:
        if case['channel'] != c:
            continue

        sensor = case['fp_device']
        
        if case['match']:
            sensor_data_pce[sensor]['match'].append(case['pce'])
            sensor_data_pearson[sensor]['match'].append(case['pearson'])
        else:
            sensor_data_pce[sensor]['mismatch'].append(case['pce'])
            sensor_data_pearson[sensor]['mismatch'].append(case['pearson'])

    sensors = sorted(sensor_data_pce.keys())

    positions_match = [i - 0.15 for i in range(len(sensors))]
    positions_mismatch = [i + 0.15 for i in range(len(sensors))]


    match_values = [sensor_data_pce[s]['match'] for s in sensors]
    mismatch_values = [sensor_data_pce[s]['mismatch'] for s in sensors]

    plt.figure(figsize=(14,6))

    vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
    for body in vp1['bodies']:
        body.set_facecolor('green')
        body.set_alpha(0.6)

    vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
    for body in vp2['bodies']:
        body.set_facecolor('red')
        body.set_alpha(0.6)

    plt.xticks(range(len(sensors)), sensors, rotation=45)
    plt.yscale('log')
    plt.xlabel("Sensor", fontsize=16)
    plt.ylabel("sPCE value", fontsize=16)
    plt.title(f"sPCE distribution per sensor - Channel {cname} PRNU", fontsize=16)

    plt.grid(True, which="both", linestyle="--", linewidth=0.5)

    plt.legend([plt.Line2D([0],[0], color='green', lw=4),
                plt.Line2D([0],[0], color='red', lw=4)],
               ['Matching', 'Mismatching'], fontsize=11)

    plt.tight_layout()

    plt.savefig(os.path.join(output_dir, f"pce_{cname}.png"), dpi=300)
    plt.close()


    match_values = [sensor_data_pearson[s]['match'] for s in sensors]
    mismatch_values = [sensor_data_pearson[s]['mismatch'] for s in sensors]

    plt.figure(figsize=(14,6))

    vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
    for body in vp1['bodies']:
        body.set_facecolor('green')
        body.set_alpha(0.6)

    vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
    for body in vp2['bodies']:
        body.set_facecolor('red')
        body.set_alpha(0.6)

    plt.xticks(range(len(sensors)), sensors, rotation=45)
    plt.xlabel("Sensor", fontsize=16)
    plt.ylabel("Pearson correlation", fontsize=16)
    plt.title(f"Pearson distribution per sensor - Channel {cname} PRNU", fontsize=16)

    plt.grid(True, linestyle="--", linewidth=0.5)

    plt.legend([plt.Line2D([0],[0], color='green', lw=4),
                plt.Line2D([0],[0], color='red', lw=4)],
               ['Matching', 'Mismatching'], fontsize=11)

    plt.tight_layout()

    plt.savefig(os.path.join(output_dir, f"pearson_{cname}.png"), dpi=300)
    plt.close()